# 128-bits of security

In [6]:
le_path = "/home/sidsabh/code/crypto/lattice-estimator"
import subprocess
import sys
sys.path.append(le_path)

In [7]:
def checkout(version):
    subprocess.run(["git", "fetch", "--all"], cwd=le_path, capture_output=True)

    if version == "stable":
        subprocess.run(["git", "checkout", "main"], cwd=le_path, capture_output=True)

    elif version == "YPIR":
        subprocess.run(["git", "checkout", "4195c66"], cwd=le_path, capture_output=True)

    elif version == "InsPIRing":
        commit = subprocess.check_output(
            ["git", "rev-list", "-1", "--before=2025-07-24", "main"],
            cwd=le_path
        ).decode().strip()

        subprocess.run(["git", "checkout", commit], cwd=le_path, capture_output=True)

In [8]:
cf = math.sqrt(2*math.pi)
class LatticeParams:
    def __init__(self, n, q, Xs, Xe, m, tag):
        self.n = n
        self.q = q
        self.Xs = Xs
        self.Xe = Xe
        self.m = m
        self.tag = tag

    def estimate(self):
        return LWE.estimate(LWEParameters(n=self.n, q=self.q, Xs=self.Xs, Xe=self.Xe, m=self.m, tag=self.tag))
    
    def estimate_rough(self):
        return LWE.estimate.rough(LWEParameters(n=self.n, q=self.q, Xs=self.Xs, Xe=self.Xe, m=self.m, tag=self.tag))


In [9]:
import importlib
import estimator

for version in ["stable", "YPIR", "InsPIRing"]:
    checkout(version)

    # force full reimport of estimator after checkout changes the source
    for mod_name in list(sys.modules.keys()):
        if mod_name.startswith("estimator"):
            del sys.modules[mod_name]
    importlib.invalidate_caches()
    import estimator
    from estimator import LWE
    from estimator.lwe_parameters import LWEParameters
    from estimator.nd import NoiseDistribution as ND
    if hasattr(ND, "DiscreteGaussian"):
        D = ND.DiscreteGaussian
    else:
        from estimator.nd import DiscreteGaussian as D


    print(f"\n{'='*60}")
    print(f"version: {version}")
    print(f"{'='*60}")

    print("First Layer")
    LatticeParams(n=1024, q=2**32, Xs=D(11), Xe=D(11), m=2**18, tag='YPIR').estimate()

    print("Second Layer")
    LatticeParams(n=2048, q=268369921*249561089, Xs=D(6.4), Xe=D(6.4), m=2**18, tag='YPIR-SP').estimate()


version: stable
First Layer
bkw                  :: rop: ≈2^338.7, m: ≈2^324.5, mem: ≈2^325.5, b: 10, t1: 0, t2: 44, ℓ: 9, #cod: 974, #top: 0, #test: 51, tag: coded-bkw
usvp                 :: rop: ≈2^130.1, red: ≈2^130.1, δ: 1.004356, β: 351, d: 2192, tag: usvp
bdd                  :: rop: ≈2^128.4, red: ≈2^128.0, svp: ≈2^126.5, β: 343, η: 376, d: 2248, tag: bdd
dual                 :: rop: ≈2^133.0, mem: ≈2^85.1, m: 1276, β: 357, d: 2300, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^134.0, red: ≈2^134.0, guess: ≈2^94.3, β: 362, p: 3, ζ: 0, t: 0, β': 372, N: ≈2^77.1, m: 1024
Second Layer
bkw                  :: rop: ≈2^524.1, m: ≈2^508.2, mem: ≈2^509.2, b: 9, t1: 0, t2: 70, ℓ: 8, #cod: 1963, #top: 1, #test: 84, tag: coded-bkw
usvp                 :: rop: ≈2^132.6, red: ≈2^132.6, δ: 1.004315, β: 356, d: 4108, tag: usvp
bdd                  :: rop: ≈2^131.5, red: ≈2^131.2, svp: ≈2^129.0, β: 351, η: 385, d: 4216, tag: bdd
dual                 :: rop: ≈2^134.5, mem: ≈2^86.2, m: ≈2^11.

In [232]:
import re
import math

sigma = 6.4
while True:
    params = LatticeParams(n=2048, q=2**64, Xs=D(6.4), Xe=D(sigma), m=2**18, tag='Word-SP')
    est = params.estimate_rough()
    print(est)
    min_rop_log2 = min(
        math.log2(float(v['rop']))
        for v in est.values()
    )
    if min_rop_log2 >= 104:
        print(f"Achieved {min_rop_log2:.1f}-bit security at sigma={sigma:.1f}")
        break
    else:
        print(f"rop=2^{min_rop_log2:.1f}, increasing sigma...")
        sigma += 100

usvp                 :: rop: ≈2^84.1, red: ≈2^84.1, δ: 1.004975, β: 288, d: 4080, tag: usvp
dual_hybrid          :: rop: ≈2^85.0, red: ≈2^85.0, guess: ≈2^71.3, β: 291, p: 3, ζ: 0, t: 0, β': 291, N: ≈2^55.6, m: ≈2^11.0
{'usvp': rop: ≈2^84.1, red: ≈2^84.1, δ: 1.004975, β: 288, d: 4080, tag: usvp, 'dual_hybrid': rop: ≈2^85.0, red: ≈2^85.0, guess: ≈2^71.3, β: 291, p: 3, ζ: 0, t: 0, β': 291, N: ≈2^55.6, m: ≈2^11.0}
rop=2^84.1, increasing sigma...
usvp                 :: rop: ≈2^93.1, red: ≈2^93.1, δ: 1.004648, β: 319, d: 4126, tag: usvp
dual_hybrid          :: rop: ≈2^94.0, red: ≈2^94.0, guess: ≈2^82.4, β: 322, p: 3, ζ: 0, t: 0, β': 322, N: ≈2^66.7, m: ≈2^11.0
{'usvp': rop: ≈2^93.1, red: ≈2^93.1, δ: 1.004648, β: 319, d: 4126, tag: usvp, 'dual_hybrid': rop: ≈2^94.0, red: ≈2^94.0, guess: ≈2^82.4, β: 322, p: 3, ζ: 0, t: 0, β': 322, N: ≈2^66.7, m: ≈2^11.0}
rop=2^93.1, increasing sigma...
usvp                 :: rop: ≈2^95.5, red: ≈2^95.5, δ: 1.004571, β: 327, d: 4144, tag: usvp
dual_hybrid     

In [106]:
print(sigma)

138.4


In [214]:
LatticeParams(n=2048, q=268369921 * 249561089, Xs=D(6.4), Xe=D(6.4), m=2**18, tag='Word-SP').estimate_rough()

usvp                 :: rop: ≈2^104.0, red: ≈2^104.0, δ: 1.004315, β: 356, d: 4108, tag: usvp
dual_hybrid          :: rop: ≈2^105.1, red: ≈2^105.1, guess: ≈2^86.9, β: 360, p: 3, ζ: 0, t: 0, β': 360, N: ≈2^71.2, m: ≈2^11.0


{'usvp': rop: ≈2^104.0, red: ≈2^104.0, δ: 1.004315, β: 356, d: 4108, tag: usvp,
 'dual_hybrid': rop: ≈2^105.1, red: ≈2^105.1, guess: ≈2^86.9, β: 360, p: 3, ζ: 0, t: 0, β': 360, N: ≈2^71.2, m: ≈2^11.0}

In [230]:
LatticeParams(n=2048, q=2**64, Xs=D(6.4), Xe=D(1762), m=2**18, tag='Word-SP').estimate()

bkw                  :: rop: ≈2^532.6, m: ≈2^516.9, mem: ≈2^517.9, b: 8, t1: 0, t2: 61, ℓ: 7, #cod: 1958, #top: 0, #test: 92, tag: coded-bkw
usvp                 :: rop: ≈2^132.6, red: ≈2^132.6, δ: 1.004315, β: 356, d: 4107, tag: usvp
bdd                  :: rop: ≈2^132.0, red: ≈2^132.0, svp: ≈2^126.0, β: 354, η: 375, d: 4066, tag: bdd
dual                 :: rop: ≈2^134.5, mem: ≈2^86.2, m: ≈2^11.1, β: 359, d: 4306, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^134.5, red: ≈2^134.5, guess: ≈2^91.8, β: 359, p: 3, ζ: 0, t: 0, β': 374, N: ≈2^76.1, m: ≈2^11.0


{'arora-gb': rop: ≈2^inf, dreg: ≈2^inf, tag: arora-gb,
 'bkw': rop: ≈2^532.6, m: ≈2^516.9, mem: ≈2^517.9, b: 8, t1: 0, t2: 61, ℓ: 7, #cod: 1958, #top: 0, #test: 92, tag: coded-bkw,
 'usvp': rop: ≈2^132.6, red: ≈2^132.6, δ: 1.004315, β: 356, d: 4107, tag: usvp,
 'bdd': rop: ≈2^132.0, red: ≈2^132.0, svp: ≈2^126.0, β: 354, η: 375, d: 4066, tag: bdd,
 'bdd_hybrid': rop: ≈2^132.3, red: ≈2^132.2, svp: ≈2^128.2, β: 354, η: 383, ζ: 0, |S|: 1, d: 4586, prob: 1, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^258.5, red: ≈2^258.5, svp: ≈2^149.9, β: 357, η: 2, ζ: 0, |S|: 1, d: 4600, prob: ≈2^-123.3, ↻: ≈2^125.5, tag: hybrid,
 'dual': rop: ≈2^134.5, mem: ≈2^86.2, m: ≈2^11.1, β: 359, d: 4306, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^134.5, red: ≈2^134.5, guess: ≈2^91.8, β: 359, p: 3, ζ: 0, t: 0, β': 374, N: ≈2^76.1, m: ≈2^11.0}

In [234]:
LatticeParams(n=2048, q=2**64, Xs=D(6.4), Xe=D(6.4*256), m=2**18, tag='Word-SP').estimate()

bkw                  :: rop: ≈2^532.6, m: ≈2^516.9, mem: ≈2^517.9, b: 8, t1: 0, t2: 61, ℓ: 7, #cod: 1958, #top: 0, #test: 92, tag: coded-bkw
usvp                 :: rop: ≈2^132.1, red: ≈2^132.1, δ: 1.004331, β: 354, d: 4259, tag: usvp
bdd                  :: rop: ≈2^131.7, red: ≈2^131.7, svp: ≈2^125.7, β: 353, η: 374, d: 4064, tag: bdd
dual                 :: rop: ≈2^134.2, mem: ≈2^85.9, m: ≈2^11.1, β: 358, d: 4306, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^134.2, red: ≈2^134.2, guess: ≈2^91.3, β: 358, p: 3, ζ: 0, t: 0, β': 373, N: ≈2^75.6, m: ≈2^11.0


{'arora-gb': rop: ≈2^inf, dreg: ≈2^inf, tag: arora-gb,
 'bkw': rop: ≈2^532.6, m: ≈2^516.9, mem: ≈2^517.9, b: 8, t1: 0, t2: 61, ℓ: 7, #cod: 1958, #top: 0, #test: 92, tag: coded-bkw,
 'usvp': rop: ≈2^132.1, red: ≈2^132.1, δ: 1.004331, β: 354, d: 4259, tag: usvp,
 'bdd': rop: ≈2^131.7, red: ≈2^131.7, svp: ≈2^125.7, β: 353, η: 374, d: 4064, tag: bdd,
 'bdd_hybrid': rop: ≈2^132.0, red: ≈2^131.9, svp: ≈2^127.9, β: 353, η: 382, ζ: 0, |S|: 1, d: 4582, prob: 1, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^251.3, red: ≈2^251.3, svp: ≈2^142.6, β: 357, η: 2, ζ: 0, |S|: 1, d: 4600, prob: ≈2^-116.1, ↻: ≈2^118.3, tag: hybrid,
 'dual': rop: ≈2^134.2, mem: ≈2^85.9, m: ≈2^11.1, β: 358, d: 4306, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^134.2, red: ≈2^134.2, guess: ≈2^91.3, β: 358, p: 3, ζ: 0, t: 0, β': 373, N: ≈2^75.6, m: ≈2^11.0}

# correctness

In [235]:
import math

class SPPackCorrectness:
    """Correctness analysis for YPIR SimplePIR.
    
    tau = q22/(2p) - (q22 % p) - (1/2)(2 + q22 % p + (q22/q2)(q2 % p)) / 2
    sigma^2 = sigma_ms^2 + sigma_scan^2 + sigma_pack^2
    delta = 2 * d * rho * exp(-pi * tau^2 / sigma^2)
    
    CDKS:      sigma_pack^2 = (d^2 - 1) * t * d * z^2 * s^2 / (4 * 3)
    InspiRING: sigma_pack^2 = t * d^2 * z^2 * s^2 / 4
    Hintless:  sigma_pack^2 = 0  (no packing noise — send resp directly)
    """
    
    # Default YPIR-SP parameters
    d  = 2048                          # poly_len
    s  = 6.4 * math.sqrt(2 * math.pi) # Gaussian width = subgaussian parameter
    p  = 1 << 15                       # RLWE plaintext modulus
    q2 = 268369921 * 249561089         # RLWE modulus (CRT product)
    q21 = 268369921                    # q_tilde_{2,1}
    q22 = 1 << 20                      # q_tilde_{2,2}
    z  = 1 << 19                       # gadget base
    t  = math.ceil(math.log(q2, z))    # gadget decomposition levels
    
    def __init__(self, l1, l2, packing='inspiring', **kwargs):
        """l1 = db_rows, l2 = db_cols, packing = 'cdks' | 'inspiring' | 'hintless'."""
        self.l1 = l1
        self.l2 = l2
        self.packing = packing
        for k, v in kwargs.items():
            setattr(self, k, v)
    
    @property
    def rho(self):
        return math.ceil(self.l2 / self.d)
    
    @property
    def pt_bits(self):
        return int(math.log2(self.p))
    
    def tau(self):
        q22, p, q2 = self.q22, self.p, self.q2
        return q22/(2*p) - (q22 % p) - 0.5 * (2 + q22 % p + (q22/q2) * (q2 % p)) / 2
    
    def sigma_pack_sq(self):
        d, s, z, t = self.d, self.s, self.z, self.t
        if self.packing == 'cdks':
            return (d**2 - 1) * t * d * z**2 * s**2 / (4 * 3)
        elif self.packing == 'inspiring':
            return t * d**2 * z**2 * s**2 / 4
        else:  # hintless
            return 0
    
    def sigma_sq_breakdown(self):
        d, s, p, q2, q21, q22, l1 = (
            self.d, self.s, self.p, self.q2, self.q21, self.q22, self.l1)
        return {
            'ms':   (q22/q21)**2 * d * s**2 / 4,
            'scan': (q22/q2)**2 * (s**2/4) * l1 * p**2,
            'pack': (q22/q2)**2 * self.sigma_pack_sq(),
        }
    
    def sigma_sq(self):
        return sum(self.sigma_sq_breakdown().values())
    
    def log2_delta(self):
        tau = self.tau()
        s2 = self.sigma_sq()
        log2_e = 1 / math.log(2)
        return math.log2(2 * self.d * self.rho) + (-math.pi * tau**2 / s2) * log2_e
    
    def __repr__(self):
        return (f"SPPackCorrectness(l1={self.l1}, l2={self.l2}, rho={self.rho}, "
                f"{self.packing}, log2(delta)={self.log2_delta():.1f})")

In [238]:
PT_BITS = {'cdks': 14, 'inspiring': 16, 'hintless': 16}

def sp_dims(num_items, item_size_bits, packing='inspiring'):
    """Compute SimplePIR DB dimensions from item count and size.
    Mirrors params_for_scenario_simplepir: instances = ceil(item_bits / (d * pt_bits))."""
    d = 2048
    pt_bits = PT_BITS[packing]
    instances = math.ceil(item_size_bits / (d * pt_bits))
    nu_1 = max(0, int(math.ceil(math.log2(max(num_items, 1)))) - 11)
    l1 = 1 << (nu_1 + 11)
    l2 = instances * d
    p = 1 << pt_bits
    return l1, l2, p

def fmt_size(nbytes):
    if nbytes >= 1 << 30:
        return f"{nbytes / (1 << 30):.1f} GiB"
    elif nbytes >= 1 << 20:
        return f"{nbytes / (1 << 20):.0f} MiB"
    elif nbytes >= 1 << 10:
        return f"{nbytes / (1 << 10):.0f} KiB"
    return f"{nbytes} B"

# Correctness table for both Ring SimplePIR and Word SimplePIR
scenarios = [
    (1 << 16, 1 << 20),     # 8 GiB
    (1 << 18, 1 << 18),     # 8 GiB
    (1 << 20, 1 << 16),     # 8 GiB
    (1 << 14, 1 << 22),     # 8 GiB
    (1 << 30, 1 << 34),     # huge
]

header = f"{'DB':>10s}  {'mode':>5s}  {'packing':>10s}  {'pt':>3s}  {'l1':>8s}  {'l2':>8s}  {'rho':>5s}  {'log2(δ)':>8s}  {'dominant':>10s}"
print(header)
print("-" * len(header))
for num_items, item_bits in scenarios:
    db_bytes = num_items * item_bits // 8
    label = fmt_size(db_bytes)
    for pack in ['cdks', 'inspiring', 'hintless']:
        l1, l2, p = sp_dims(num_items, item_bits, pack)
        ring = SPPackCorrectness(l1=l1, l2=l2, packing=pack, p=p)
        word = WordPIRCorrectness(l1=l1, l2=l2, packing=pack, p=p)
        for mode, c in [("ring", ring), ("word", word)]:
            bd = c.sigma_sq_breakdown()
            dominant = max(bd, key=bd.get)
            print(f"{label:>10s}  {mode:>5s}  {pack:>10s}  {c.pt_bits:>3d}  {l1:>8d}  {l2:>8d}  {c.rho:>5d}  {c.log2_delta():>8.1f}  {dominant:>10s}")
    print()

        DB   mode     packing   pt        l1        l2    rho   log2(δ)    dominant
-----------------------------------------------------------------------------------
   8.0 GiB   ring        cdks   14     65536     75776     37     -97.4        pack
   8.0 GiB   word        cdks   14     65536     75776     37     -97.4        pack
   8.0 GiB   ring   inspiring   16     65536     65536     32    -106.4          ms
   8.0 GiB   word   inspiring   16     65536     65536     32    -106.4          ms
   8.0 GiB   ring    hintless   16     65536     65536     32    -109.7          ms
   8.0 GiB   word    hintless   16     65536     65536     32    -109.7          ms

   8.0 GiB   ring        cdks   14    262144     20480     10     -99.3        pack
   8.0 GiB   word        cdks   14    262144     20480     10     -99.3        pack
   8.0 GiB   ring   inspiring   16    262144     16384      8    -108.4          ms
   8.0 GiB   word   inspiring   16    262144     16384      8    -108.4    

In [239]:
import math

W = 1 << 64  # word modulus

class WordPIRCorrectness:
    """Word SimplePIR correctness — client samples e_W ~ D(sigma_W) directly in Z_W.
    
    sigma_W = (W/Q) * sigma_2.  Server modswitches Z_W -> Z_Q, scaling scan by Q/W.
    Since (Q/W) * sigma_W = sigma_2, scan is identically ring-based.
    Only new term vs SPPackCorrectness: hint rounding (H_W -> H_Q modswitch x secret).
    Security: D(6.4 * W/Q) ≈ D(1762) verified > 128-bit.
    """
    
    d  = 2048
    n_lwe = 2048
    s  = 6.4 * math.sqrt(2 * math.pi)               # sigma_2 (RLWE width)
    p  = 1 << 15
    q2 = 268369921 * 249561089
    q21 = 268369921
    q22 = 1 << 20
    z  = 1 << 19
    t  = math.ceil(math.log(q2, z))    # gadget decomposition levels
    
    def __init__(self, l1, l2, packing='inspiring', **kwargs):
        self.l1 = l1
        self.l2 = l2
        self.packing = packing
        for k, v in kwargs.items():
            setattr(self, k, v)
    
    @property
    def rho(self):
        return math.ceil(self.l2 / self.d)
    
    @property
    def pt_bits(self):
        return int(math.log2(self.p))
    
    def tau(self):
        q22, p, q2 = self.q22, self.p, self.q2
        return q22/(2*p) - (q22 % p) - 0.5 * (2 + q22 % p + (q22/q2) * (q2 % p)) / 2
    
    def sigma_pack_sq(self):
        d, s, z, t = self.d, self.s, self.z, self.t
        if self.packing == 'cdks':
            return (d**2 - 1) * t * d * z**2 * s**2 / (4 * 3)
        elif self.packing == 'inspiring':
            return t * d**2 * z**2 * s**2 / 4
        else:
            return 0
    
    def sigma_sq_breakdown(self):
        d, s, p, q2, q21, q22, l1, n = (
            self.d, self.s, self.p, self.q2, self.q21, self.q22, self.l1, self.n_lwe)
        return {
            'ms':   (q22/q21)**2 * d * s**2 / 4,
            'scan': (q22/q2)**2 * (s**2/4) * l1 * p**2,               # = ring-based (Q/W cancels W/Q in sigma_W)
            'pack': (q22/q2)**2 * self.sigma_pack_sq(),
            'hint': (q22/q2)**2 * n * s**2 / 4,                       # H_W -> H_Q rounding x secret
        }
    
    def sigma_sq(self):
        return sum(self.sigma_sq_breakdown().values())
    
    def log2_delta(self):
        tau = self.tau()
        s2 = self.sigma_sq()
        log2_e = 1 / math.log(2)
        return math.log2(2 * self.d * self.rho) + (-math.pi * tau**2 / s2) * log2_e
    
    def __repr__(self):
        return (f"WordPIRCorrectness(l1={self.l1}, l2={self.l2}, rho={self.rho}, "
                f"{self.packing}, log2(delta)={self.log2_delta():.1f})")

# Quick check
eff_std_dev = 6.4 * W / (268369921 * 249561089)
print(f"sigma_W = (W/Q) * sigma_2,  effective std dev = 6.4 * W/Q = {eff_std_dev:.1f}")
print(f"D({eff_std_dev:.0f}) verified > 128-bit security")
print()

l1, l2, p = sp_dims(1 << 16, 491520, 'inspiring')
ring = SPPackCorrectness(l1, l2, 'inspiring', p=p)
word = WordPIRCorrectness(l1, l2, 'inspiring', p=p)
print(f"Ring: {ring}")
print(f"Word: {word}")
print()
for label, obj in [("Ring", ring), ("Word", word)]:
    s2 = obj.sigma_sq()
    print(f"{label}: log2(sigma^2) = {math.log2(s2):.1f}")
    for k, v in obj.sigma_sq_breakdown().items():
        pct = v / s2 * 100
        print(f"  {k:>6}: log2 = {math.log2(v) if v > 0 else float('-inf'):>8.1f}  ({pct:5.1f}%)")
    print()

sigma_W = (W/Q) * sigma_2,  effective std dev = 6.4 * W/Q = 1762.7
D(1763) verified > 128-bit security

Ring: SPPackCorrectness(l1=65536, l2=30720, rho=15, inspiring, log2(delta)=-107.5)
Word: WordPIRCorrectness(l1=65536, l2=30720, rho=15, inspiring, log2(delta)=-107.5)

Ring: log2(sigma^2) = 1.0
      ms: log2 =      1.0  ( 97.4%)
    scan: log2 =    -17.8  (  0.0%)
    pack: log2 =     -4.2  (  2.6%)

Word: log2(sigma^2) = 1.0
      ms: log2 =      1.0  ( 97.4%)
    scan: log2 =    -17.8  (  0.0%)
    pack: log2 =     -4.2  (  2.6%)
    hint: log2 =    -54.8  (  0.0%)

